# LLM From Scratch — Colab Runner

Run this notebook via the **Colaboratory** VS Code extension or directly on [colab.research.google.com](https://colab.research.google.com).

**How it works:**
- This notebook runs on Google's remote runtime — your local VS Code files are not accessible there.
- Code is pulled from GitHub on each session. Edit locally → push to GitHub → re-run Cell 2 to pick up changes.
- Weights and checkpoints are saved to Google Drive so they survive runtime restarts.

In [2]:
# Cell 1 — Mount Google Drive
# Weights, checkpoints, and cached HF models are saved here and persist across sessions.
from google.colab import drive
drive.mount('/content/drive')

In [3]:
# Cell 2 — Clone repo on first run, pull latest on subsequent runs
# The Colab runtime runs on Google's servers — your local files are not available here.
# Always push changes to GitHub first, then re-run this cell to pick them up.
import os, shutil

REPO_URL  = "https://github.com/<your-username>/llmFromScratch"
REPO_DIR  = "/content/llmFromScratch"

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

# Clear bytecode cache so stale .pyc files cannot shadow updated source code.
!find {REPO_DIR} -type d -name __pycache__ -exec rm -rf {} + 2>/dev/null; true

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [4]:
# Cell 3 — Install dependencies
!pip install -r requirements.txt -q

In [5]:
# Cell 4 — (Gemma3 only) Authenticate with HuggingFace
# Skip this cell if you are only training GPT-2 models.
# Required once per session — token is not stored between runtimes.
# from huggingface_hub import login
# login()  # Paste your HF token when prompted

# Use this to use Google Colab Secret Tokens, if Hugging Face Token is saved
from google.colab import userdata
from huggingface_hub import login
hf_token = userdata.get("hf_token")
login(token=hf_token)

In [ ]:
# Cell 5 — Train
# Weights are saved to MyDrive/llmFromScratch/fine_tuned_weights/ automatically.
# Edit the flags below as needed — see README.md for the full CLI reference.
!python -m src.main \
    --model "gpt2-small (124M)" \
    --epochs 3 \
    --lr 4e-4 \
    --warmup-steps 100

In [ ]:
# Cell 6 — (Optional) LoRA fine-tuning
!python -m src.main \
    --model "gemma3-1b" \
    --batch-size 4 \      
    --grad-accum 2 \      
    --epochs 5 \          
    --lr 2e-4 \           
    --warmup-steps 50 \
    --lora \
    --lora-rank 8 \
    --lora-alpha 16

In [ ]:
# Cell 7 — Plot training diagnostics
import json
import sys
sys.path.insert(0, REPO_DIR)

from src.utils.plotting import plot_loss_curves
from src.models.registry import load_metrics

MODEL = "gpt2-small (124M)"   # change to match the model you trained

metrics = load_metrics(MODEL)
if metrics:
    fig = plot_loss_curves(metrics)
    fig.savefig("training_diagnostics.png", dpi=150, bbox_inches='tight')
    print("Saved training_diagnostics.png")
else:
    print(f"No metrics found for '{MODEL}'")